# CrashDiag GRPO — independent Kaggle notebook

This notebook starts in a fresh Kaggle GPU session. It downloads and hash-verifies the datasets and completed SFT adapter from the private `devaanshpa/CrashDiag` Storage Bucket, probes the authenticated Vultr sandbox at `https://sandbox.devaanshpathak.com`, runs mechanically rewarded GRPO, uploads every retained checkpoint/final artifact, and optionally performs held-out mechanical evaluation.

Enable **GPU** and **Internet**. Attach Kaggle Secrets named `HF_TOKEN` and `CRASHDIAG_SANDBOX_TOKEN`. Paste the exact `RUN_ID` and `SOURCE_COMMIT` printed by `notebooks/sft.ipynb`; this notebook never falls back to an untrained base model when SFT artifacts are missing.

In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys

REPO_URL = "https://github.com/Indium-AI-Labs/CrashDiag.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/kaggle/working/CrashDiag")
BUCKET_ID = "devaanshpa/CrashDiag"
SANDBOX_URL = "https://sandbox.devaanshpathak.com"
RUN_ID = os.environ.get("CRASHDIAG_RUN_ID") or "PASTE_SFT_RUN_ID_HERE"
SOURCE_COMMIT = os.environ.get("CRASHDIAG_SOURCE_COMMIT") or "PASTE_SFT_SOURCE_COMMIT_HERE"
MODE = "smoke"  # use smoke first; change to full only after rewards/backend look correct
PRECISION = "auto"
SEED = 42
NUM_GENERATIONS = 4
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
SMOKE_STEPS = 20
RESUME_LATEST_CHECKPOINT = False  # opt in only after inspecting a partial checkpoint
EPISODES_PER_FAULT = 10

if RUN_ID == "PASTE_SFT_RUN_ID_HERE":
    raise ValueError("Set RUN_ID to the exact value printed by notebooks/sft.ipynb")
if SOURCE_COMMIT == "PASTE_SFT_SOURCE_COMMIT_HERE":
    raise ValueError("Set SOURCE_COMMIT to the exact value printed by notebooks/sft.ipynb")
if re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT) is None:
    raise ValueError("SOURCE_COMMIT must be the full 40-character lowercase Git SHA printed by SFT")
if MODE not in {"smoke", "full"}:
    raise ValueError("MODE must be 'smoke' or 'full'")
GRPO_STAGE = "grpo-smoke" if MODE == "smoke" else "grpo"
OUTPUT_DIR = REPO_DIR / "outputs" / GRPO_STAGE
MAX_STEPS = SMOKE_STEPS if MODE == "smoke" else -1
SAVE_STEPS = 10 if MODE == "smoke" else 25

print(f"RUN_ID={RUN_ID}")
print(f"SOURCE_COMMIT={SOURCE_COMMIT}")
print(f"mode={MODE}, stage={GRPO_STAGE}, max_steps={MAX_STEPS}")
print(f"sandbox={SANDBOX_URL}")

## Install the repository in this fresh session

The notebook does not rely on files or imports left by the SFT kernel.

In [ ]:
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "merge", "--ff-only", f"origin/{REPO_BRANCH}"], check=True)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"Refusing to overwrite non-repository directory: {REPO_DIR}")
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", SOURCE_COMMIT], check=True)
CURRENT_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if CURRENT_COMMIT != SOURCE_COMMIT:
    raise RuntimeError(f"Git checkout mismatch: expected {SOURCE_COMMIT}, got {CURRENT_COMMIT}")

# TorchAO is optional and CrashDiag does not use it. Some Kaggle images ship
# an old build that makes current PEFT fail while dispatching ordinary LoRA.
torchao_probe = subprocess.run(
    [sys.executable, "-m", "pip", "show", "torchao"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=False,
)
if torchao_probe.returncode == 0:
    print("removing unused preinstalled torchao to avoid a PEFT version conflict")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train]"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"checked_out_source_commit={CURRENT_COMMIT}")

## Load both secrets without writing them to disk

`HF_TOKEN` needs write access to the private artifact bucket. `CRASHDIAG_SANDBOX_TOKEN` must match the token in the Vultr host's `.env`.

In [ ]:
def required_secret(name: str) -> str:
    existing = os.environ.get(name)
    if existing:
        return existing
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception as exc:
        raise RuntimeError(f"Attach the Kaggle Secret {name!r} before continuing") from exc
    if not value:
        raise RuntimeError(f"Kaggle Secret {name!r} is empty")
    return value

os.environ["HF_TOKEN"] = required_secret("HF_TOKEN")
os.environ["CRASHDIAG_SANDBOX_TOKEN"] = required_secret("CRASHDIAG_SANDBOX_TOKEN")
os.environ["CRASHDIAG_SANDBOX_URL"] = SANDBOX_URL
os.environ["CRASHDIAG_HF_BUCKET_ID"] = BUCKET_ID
os.environ["CRASHDIAG_ARTIFACT_PREFIX"] = "runs"
os.environ["CRASHDIAG_ARTIFACT_LOCAL_ROOT"] = str(REPO_DIR / "artifacts")
os.environ["CRASHDIAG_ARTIFACT_UPLOAD_POLICY"] = "required"
os.environ["CRASHDIAG_RUN_ID"] = RUN_ID
print("Kaggle Secrets loaded into the process environment (values hidden).")

## Restore the SFT handoff from the private bucket

A run-level success marker is intentionally absent until full GRPO and evaluation finish, so recovery explicitly allows an incomplete run. Completed dataset and SFT stage manifests are still hash-verified. An existing local recovery directory is verified instead of overwritten.

In [ ]:
from training.artifacts import ArtifactConfig, ArtifactUploader

artifact_config = ArtifactConfig(
    bucket_id=BUCKET_ID,
    run_id=RUN_ID,
    prefix="runs",
    policy="required",
    local_root=REPO_DIR / "artifacts",
    token=os.environ["HF_TOKEN"],
)
uploader = ArtifactUploader(artifact_config)
RUN_DIR = artifact_config.local_run_root
if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
    uploader.verify_local_run(RUN_DIR, require_complete=False)
    print(f"verified existing recovery directory: {RUN_DIR}")
else:
    uploader.download_run(RUN_DIR, allow_incomplete=True)
    print(f"downloaded and verified: {uploader.remote_uri()}")

SFT_DIR = RUN_DIR / "sft"
DATASET_DIR = RUN_DIR / "datasets"
required_handoff = [
    SFT_DIR / "adapter_config.json",
    SFT_DIR / "manifest.json",
    SFT_DIR / "_SUCCESS.json",
    DATASET_DIR / "grpo_train.jsonl",
    DATASET_DIR / "grpo_eval.jsonl",
    DATASET_DIR / "manifest.json",
    DATASET_DIR / "_SUCCESS.json",
]
missing = [str(path) for path in required_handoff if not path.is_file()]
if missing:
    raise RuntimeError(f"Refusing to train without completed SFT/dataset artifacts: {missing}")
for manifest_path in (SFT_DIR / "manifest.json", DATASET_DIR / "manifest.json"):
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    artifact_commit = manifest.get("runtime", {}).get("git_commit")
    if artifact_commit != CURRENT_COMMIT:
        raise RuntimeError(
            f"Artifact/code revision mismatch for {manifest_path.parent.name}: "
            f"artifact={artifact_commit!r}, notebook={CURRENT_COMMIT!r}"
        )

## Probe Vultr and the Kaggle GPU before loading weights

The probe creates an authenticated disposable session through Caddy, checks service and application health, and deletes the session on exit.

In [ ]:
from crashdiag.sandbox_apps.http import HttpSandbox

with HttpSandbox(
    SANDBOX_URL,
    api_token=os.environ["CRASHDIAG_SANDBOX_TOKEN"],
    timeout=15.0,
) as sandbox:
    service = sandbox.service_health()
    application = sandbox.health_check()
if service.get("status") != "ok":
    raise RuntimeError(f"Vultr service preflight failed: {service}")
if application.get("healthy") is not True:
    raise RuntimeError(f"Vultr sandbox preflight failed: {application}")
print("Vultr HTTPS/auth/session preflight passed.")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is visible. Enable a GPU in Kaggle Notebook Settings.")
print(f"torch={torch.__version__}")
print(f"gpu={torch.cuda.get_device_name(0)}")
print(f"bf16_supported={torch.cuda.is_bf16_supported()}")

## Select a resumable checkpoint

A completed stage is protected by its manifest and success marker and cannot be silently overwritten. Partial checkpoints are synced for recovery but do not yet have per-checkpoint success manifests, so automatic resume is off by default. Set `RESUME_LATEST_CHECKPOINT = True` only after inspecting an interrupted stage and accepting that weaker integrity boundary.

In [ ]:
REMOTE_STAGE_DIR = RUN_DIR / GRPO_STAGE
checkpoints = []
if REMOTE_STAGE_DIR.is_dir():
    checkpoints = sorted(
        (path for path in REMOTE_STAGE_DIR.glob("checkpoint-*") if path.is_dir()),
        key=lambda path: int(path.name.rsplit("-", 1)[1]),
    )
RESUME_FROM = checkpoints[-1] if RESUME_LATEST_CHECKPOINT and checkpoints else None
print(f"resume_from={RESUME_FROM}")

## Run mechanically rewarded GRPO

Start with `MODE = "smoke"`. Every completion is parsed into one bounded action, replayed against its exact seeded scenario on Vultr, and receives `1.0` only if executable fault resolution and health checks pass. There is no LLM grader. The per-device batch is one for Kaggle memory safety; four-step accumulation keeps the generation batch divisible by four generations.

In [ ]:
from training.grpo import main as grpo_main

grpo_args = [
    "--model", str(SFT_DIR),
    "--train-file", str(DATASET_DIR / "grpo_train.jsonl"),
    "--eval-file", "",
    "--output-dir", str(OUTPUT_DIR),
    "--artifact-stage", GRPO_STAGE,
    "--precision", PRECISION,
    "--batch-size", str(BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION_STEPS),
    "--num-generations", str(NUM_GENERATIONS),
    "--max-steps", str(MAX_STEPS),
    "--epochs", "1",
    "--save-steps", str(SAVE_STEPS),
    "--seed", str(SEED),
    "--report-to", "none",
]
if RESUME_FROM is not None:
    grpo_args.extend(["--resume-from-checkpoint", str(RESUME_FROM)])
grpo_main(grpo_args)

stage_success = REPO_DIR / "artifacts" / RUN_ID / GRPO_STAGE / "_SUCCESS.json"
if not stage_success.is_file():
    raise RuntimeError(f"GRPO stage did not produce its success marker: {stage_success}")
print(f"GRPO stage uploaded: {uploader.remote_uri(GRPO_STAGE)}")

## Display the signed GRPO training report

The available loss, learning-rate, mechanical-reward, and policy-diagnostic series are rendered directly from numeric Trainer logs. Missing series are omitted rather than invented. JSON, Markdown, and SVG files are created before the GRPO stage manifest and success marker, so the displayed files are already hash-covered in the private bucket.

In [ ]:
from IPython.display import SVG, display

def display_signed_report(report_dir: Path, summary_name: str, manifest_path: Path, manifest_prefix: str, remote_uri: str) -> None:
    summary_path = report_dir / summary_name
    charts = sorted(report_dir.glob("*.svg"))
    report_files = sorted(path for path in report_dir.iterdir() if path.is_file()) if report_dir.is_dir() else []
    if not summary_path.is_file() or not charts or not report_files:
        raise RuntimeError(f"Training report is incomplete: {report_dir}")
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print(json.dumps(summary, indent=2, sort_keys=True))
    for chart in charts:
        print(f"displaying {chart.name}")
        display(SVG(filename=str(chart)))
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    manifest_paths = {entry["path"] for entry in manifest.get("files", [])}
    expected = {manifest_prefix + path.name for path in report_files}
    missing = sorted(expected - manifest_paths)
    if missing:
        raise RuntimeError(f"Displayed report files are absent from the signed bucket manifest: {missing}")
    print(f"signed report files uploaded: {remote_uri}")

GRPO_REPORT_DIR = OUTPUT_DIR / "reports"
display_signed_report(
    GRPO_REPORT_DIR,
    "metrics_summary.json",
    RUN_DIR / GRPO_STAGE / "manifest.json",
    "reports/",
    uploader.remote_uri(GRPO_STAGE) + "/reports",
)

## Full-mode held-out evaluation and run completion

This cell intentionally does nothing after a smoke run. After changing `MODE` to `full` and rerunning in a fresh session with the same `RUN_ID`, it executes ten held-out episodes per fault, displays mechanically verified per-fault success rates, uploads the complete evaluation plus its signed report files, verifies all stage markers, and writes the run-level success marker last.

In [ ]:
if MODE == "full":
    from training.evaluate import main as evaluate_main

    evaluate_main([
        "--model", str(OUTPUT_DIR),
        "--sandbox-url", SANDBOX_URL,
        "--episodes-per-fault", str(EPISODES_PER_FAULT),
        "--precision", PRECISION,
        "--output", "outputs/evaluation.json",
    ])
    display_signed_report(
        REPO_DIR / "outputs/evaluation-report",
        "mechanical_evaluation_summary.json",
        RUN_DIR / "evaluation/manifest.json",
        "",
        uploader.remote_uri("evaluation"),
    )
    uploader.complete_run({"stages": ["datasets", "sft", "grpo", "evaluation"]})
    print(f"complete artifact run: {uploader.remote_uri()}")
else:
    print("Smoke stage complete. Inspect rewards and backend-error logs, then set MODE='full' for the full job.")